In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [2]:
import tensorflow as tf

print(tf.test.is_built_with_cuda())
print(tf.config.list_physical_devices('GPU'))
print(tf. __version__)

True
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
2.11.0


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../dataset_generation/dataset_130k.csv')

In [4]:
# df = df.iloc[0:10, :] 
df

,algorithm,encrypt
0,vigenere,qlhskblfzvzsbumzsnifkvvhgplmw
1,substitution,bjjnjnsjamfnaasjjudjascnsajfjjgncassca
2,caesar,ignlwpigtetcipcrtrpclwpirpcxicdi
3,vigenere,losqhcfjlvzybugusrrjvyywdipygdglxzygmznyb
4,caesar,mehjxojeruqhurubvehjejxqj
...,...,...
133875,substitution,vecaamqecodhemvdvajcacdhmedhqeccqd
133876,affine,yfbylksinybj
133877,rail_fence,dienglohattyeictnneohtmiilhlehva
133878,affine,ldbbbtjhllbbtvhpbhttpvxbtldbhbhbvhzxzzj


In [5]:
df.algorithm.unique()

array(['vigenere', 'substitution', 'caesar', 'affine', 'rail_fence'],
      dtype=object)

In [6]:
# Step 1 -> convert text to numbers using textVocab
from tensorflow.keras.layers import TextVectorization

vectorize_layer = TextVectorization(
    max_tokens=100,
    split="character",
    output_mode = 'int',
)

vectorize_layer.adapt(df.encrypt.to_list())

In [7]:
vectorized_strings = np.array([vectorize_layer(s).numpy() for s in df['encrypt']], dtype='object')
vectorized_strings.shape

(133880,)

In [8]:
images = np.array(
    [[np.roll(vector, idx) for idx in range(len(vector))] for vector in vectorized_strings]
, dtype='object')
images.shape

(133880,)

In [9]:
import cv2

X = np.array(
    [cv2.resize(np.array(image, dtype='uint8'), dsize=(64,64)) for image in images]
)
X.shape

(133880, 64, 64)

In [10]:
X = X.reshape(-1,64,64,1)
X.shape

(133880, 64, 64, 1)

In [11]:
input_shape = X[0].shape
input_shape

(64, 64, 1)

In [12]:
y = pd.get_dummies(df[['algorithm']])

In [13]:
X.shape

(133880, 64, 64, 1)

In [14]:
y.shape

(133880, 5)

In [15]:
output_shape = y.shape[1]

In [16]:
from sklearn.model_selection import train_test_split
x_train, x_rem, y_train, y_rem = train_test_split(X, y, test_size=0.2, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_rem, y_rem, test_size=0.2, random_state=42)

In [17]:
x_train = np.asarray(x_train)
y_train = np.asarray(y_train)
x_val = np.asarray(x_val)
y_val = np.asarray(y_val)
x_test = np.asarray(x_test)
y_test = np.asarray(y_test)

# Training

In [18]:
from tensorflow.keras.layers import Conv2D, \
BatchNormalization, MaxPool2D, Flatten, Dense, Dropout

model = tf.keras.models.Sequential()

model.add(tf.keras.Input(shape=input_shape))

model.add(Conv2D(512, (3,3), activation='relu'))
model.add(MaxPool2D())
model.add(BatchNormalization())

model.add(Dropout(0.1))
model.add(Conv2D(2048, (3,3), activation='relu'))
model.add(MaxPool2D())
model.add(BatchNormalization())

model.add(Dropout(0.1))
model.add(Conv2D(8152, (3,3), activation='relu'))
model.add(MaxPool2D())
model.add(BatchNormalization())

model.add(Flatten())

model.add(Dropout(0.1))
model.add(Dense(units=512, activation='relu'))
model.add(BatchNormalization())

model.add(Dropout(0.1))
model.add(Dense(units=128, activation='relu'))
model.add(BatchNormalization())

model.add(Dropout(0.1))
model.add(Dense(units=32, activation='relu'))
model.add(BatchNormalization())

model.add(Dense(units=output_shape, activation='softmax'))

In [19]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 62, 62, 512)       5120      
                                                                 
 max_pooling2d (MaxPooling2D  (None, 31, 31, 512)      0         
 )                                                               
                                                                 
 batch_normalization (BatchN  (None, 31, 31, 512)      2048      
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 31, 31, 512)       0         
                                                                 
 conv2d_1 (Conv2D)           (None, 29, 29, 2048)      9439232   
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 14, 14, 2048)     0

In [20]:
# ! export XLA_FLAGS=--xla_gpu_cuda_data_dir=$CONDA_PREFIX

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(
    loss="categorical_crossentropy", metrics=["accuracy"]
)
callback = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True
    )
]

history = model.fit(
    x_train,
    y_train,
    epochs=10000,
    validation_data=(x_val, y_val),
    callbacks=callback,
)

Epoch 1/10000
3347/3347 [==============================] - 1018s 300ms/step - loss: 1.1354 - accuracy: 0.5025 - val_loss: 1.2613 - val_accuracy: 0.4521
Epoch 2/10000
3347/3347 [==============================] - 999s 298ms/step - loss: 1.0398 - accuracy: 0.5523 - val_loss: 1.1702 - val_accuracy: 0.4911
Epoch 3/10000
3347/3347 [==============================] - 997s 298ms/step - loss: 0.9796 - accuracy: 0.5853 - val_loss: 1.3534 - val_accuracy: 0.3855
Epoch 4/10000
3347/3347 [==============================] - 997s 298ms/step - loss: 0.9336 - accuracy: 0.6075 - val_loss: 1.3609 - val_accuracy: 0.4859
Epoch 5/10000
3347/3347 [==============================] - 998s 298ms/step - loss: 0.8968 - accuracy: 0.6251 - val_loss: 0.8948 - val_accuracy: 0.6214
Epoch 6/10000
3347/3347 [==============================] - 999s 298ms/step - loss: 0.8549 - accuracy: 0.6437 - val_loss: 0.8883 - val_accuracy: 0.6262
Epoch 7/10000
3347/3347 [==============================] - 998s 298ms/step - loss: 0.8089 - a

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
# model.save('./model_5_class_no_rotate')

In [ ]:
test_loss, test_acc = model.evaluate(x_val, y_val)
print(f"test_loss={test_loss}, test_acc={test_acc}")